In [ ]:
import sys
import os
import time as _time
import pandas as pd
import inspect
from pathlib import Path


 
# Find the project root
def find_project_root(marker="src/data_prep.py"):

    # Calls the class working directory from pathlib, and captures the starting location
    here = Path.cwd()

    # Creates an upward search path, checking one folder layer at a time until
    # candidate (the project root) is returned 
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
        
    # Incase the file is not found
    raise FileNotFoundError("Could not find the file")

# Establishing the project paths and root
project_root = find_project_root()
src_path = project_root / "src"
data_path = project_root / "data"



print(f"Project root confirmed at: {project_root}")

# Insert src directory at top priority
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import the processing function
from data_prep import build_full_dataset
from data_prep import fetch_sequence
from data_prep import build_ClinVar_dataset 

#Sanity checks to see if the function is actually returning the DNA sequence
print(fetch_sequence("14", 23412740))       


ClinVar_path = str(data_path / "raw" / "variant_summary.txt.gz")

#start = _time.time()
#test_df = build_ClinVar_dataset(ClinVar_path, "test_out.csv", limit=500)
#elapsed = _time.time() - start

#print(f"{elapsed:.1f}s for {len(test_df)} rows kept")

# Defining the ClinVar file path (includes lots of variants, which are filtered in data_prep)


# Unlike ClinVar, gnomAD includes seperated data for each gene. This takes
# the paths for each of the three genes.
gnomAD_csv_paths = {
    "MYH7": str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN": str(data_path / "raw" / "gnomAD_TTN.csv")
}
out_path = str(data_path / "processed" / "dataset.csv")

# Runs data preparation pipeline and calls build_ClinVar_dataset from data_prep
# note that build_full_dataset was originally ran, but for the second time running, because
# I already loaded all the benign gnomAD variants, only build_ClinVar_dataset is neccesary
print("Starting data preparation pipeline")

final_df = build_ClinVar_dataset(
    ClinVar_path=ClinVar_path,
    out_path=out_path,
    old_frac=0.5,   # this part was already saved
    new_frac=1.0,   # new parts that need to be added
)

# Configure Pandas display options and show results
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


print("\n--- ENTIRE FINISHED DATASET ---")
display(final_df)

In [ ]:
import sys, json, time
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path     = project_root / "src"
results_path = project_root / "results"

# Puts src first in the search list, always, and reloads explain so edits are picked up
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))
sys.modules.pop("explain", None)
import explain
print("using", explain.__file__)

# Setup and Gate 0 (the saved models load and give the saved test predictions)
start = time.time()
matched = pd.read_csv(project_root / "data" / "processed" / "matched_seed42.csv")
assert int(matched["pos"].sum()) == 268677163703, "matched_seed42.csv is not the file the models were trained on"
setup = explain.final_setup(matched)
models = explain.load_final_models(matched, setup)

# Inputs for IG on the 343 test variants
test_df = setup["test_df"].reset_index(drop = True)
F_test = setup["F_test"]
names = list(setup["raw"](test_df).columns)
typical = explain.typical_benign(setup["F_fit"], setup["fit_df"])
X_test = explain.encode_dataset(test_df["sequence"].tolist())
x_base, f_base = explain.baselines(test_df, F_test, typical)

# Gate 1: completeness
dna, feat, p, p0, delta = explain.integrated_gradients(models, X_test, F_test, x_base, f_base)
gap = explain.check_completeness(dna, feat, p, p0, delta)

# Gates 2 to 6
explain.check_linearity(models, X_test, F_test, x_base, f_base)
same_sign, r = explain.check_ablation(models, X_test, F_test, f_base, feat)
rand = explain.check_randomization(models, X_test, F_test, x_base, f_base, feat, setup["fit_df"], setup["F_fit"])
planted = explain.check_planted_feature(setup["fit_df"], setup["F_fit"], test_df, F_test)
share, peak = explain.check_planted_motif(setup["fit_df"], test_df, F_test.shape[1])

# Saves the gate results
gates = {"completeness_largest_gap": float(gap.max()), "ablation_same_sign": float(same_sign), "ablation_r": float(r),
         "randomization": {k: list(v) for k, v in rand.items()}, "planted_feature": planted,
         "planted_motif_share": share, "planted_motif_peak": peak}
json.dump(gates, open(results_path / "ig_gates.json", "w"), indent = 2)

# Face validity: saved explanations for the test variants, averaged by reason, kind and class
ig_test = explain.explain_saved(models, test_df, F_test, typical, names, str(results_path / "ig_test.csv"))
face = explain.groups_mean(ig_test, test_df)
face.to_csv(results_path / "face_validity.csv")
pd.set_option("display.width", 250)
print(face.to_string())
print(f"All gates passed in {time.time() - start:.0f} s")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from train import cap_benign 
import Revel_cadd_alphamissense_bench

data_path = project_root/"data"
dataset_path = data_path/"processed"/"dataset.csv"
# Reads the dataset_path, same seed for reproducibility
df = pd.read_csv(dataset_path)
print(dataset_path)
seed = 42

# Calls cap_benign from train to cap the benign variants amount
capped = cap_benign(df, max_benign=None, seed=seed)

# Makes the datasets also inside of train, uses data stratification
rest_df, demo_df = train_test_split(capped, test_size= 0.02, stratify=capped["label"], random_state=seed)
train_df, test_df = train_test_split(rest_df, test_size=0.10/0.98, stratify=rest_df["label"], random_state=seed)

# Prints the value count of the amount of rows and each label
print(f"test_df = {len(test_df)} rows")
print(test_df["label"].value_counts())

# Calls build_annotation_subset from annotate.py
subset = Revel_cadd_alphamissense_bench.build_annotation_subset(df, test_df)

# Calls annotate_dataset from annotate.py, with no in_path, and a csv output. 
annotated = Revel_cadd_alphamissense_bench.annotate_dataset (None, str(data_path / "processed" / "dataset_annotated.csv"), df=subset)

results = Revel_cadd_alphamissense_bench.revel_cadd_benchmark(annotated, test_df)

# Returns the results by using .items() to return a (key, value) pair each time
print("\n--- Binary Pathogenic-vs-Benign AUC-ROC, same 716 test variants ---")
for name, auc in results.items():
    print(f"{name:35s} {auc:.3f}")


In [ ]:
import sys
from pathlib import Path
import pandas as pd
here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "src/data_prep.py").exists())
if str(project_root / "src") in sys.path:
    sys.path.remove(str(project_root / "src"))
sys.path.insert(0, str(project_root / "src"))
sys.modules.pop("features", None)
import features

matched = pd.read_csv(project_root / "data" / "processed" / "matched_seed42.csv")
ann = features.annotate(matched)
span = {g: (d["pos"].min(), d["pos"].max()) for g, d in matched.groupby("gene")}

X1 = features.build(matched, ann = ann)                     # gene_span not passed
X2 = features.build(matched, ann = ann, gene_span = span)   # passed, like train.py does
print(X1.shape, X2.shape)
print(X2.loc[[0, 717, 1406, 1403], ["rel_pos", "tx_protein_frac", "tx_intron_dist", "ttn_psi_dcm", "ttn_in_novex3", "aa_grantham", "aa_blosum"]].round(3))

In [ ]:

import sys
import time
from pathlib import Path
import pandas as pd


def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")


project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project root :", project_root)
print("src on path  :", str(src_path) in sys.path)

%load_ext autoreload
%autoreload 2

clinvar_dataset_path = str(data_path / "processed" / "dataset.csv")
clinvar_raw_path     = str(data_path / "raw" / "variant_summary.txt.gz")
out_path             = str(data_path / "processed" / "dataset_full.csv")
smoke_path           = str(data_path / "processed" / "SMOKE_TEST.csv")

gnomAD_csv_paths = {
    "MYH7":   str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN":    str(data_path / "raw" / "gnomAD_TTN.csv"),
}

for name, p in [("clinvar_dataset", clinvar_dataset_path),
                ("clinvar_raw", clinvar_raw_path),
                *gnomAD_csv_paths.items()]:
    print(f"{name:16s} exists: {Path(p).exists()}")

from data_prep import consequence_targets
from consequence import parse_consequence

df = pd.read_csv(clinvar_dataset_path)
print("rows:", len(df))
print("columns:", list(df.columns), "\n")

if "consequence" not in df.columns:
    df["consequence"] = df["name"].apply(parse_consequence)

print(pd.crosstab([df["gene"], df["consequence"]], df["label"]).to_string(), "\n")

TARGETS = {}
for g in ["MYH7", "MYBPC3", "TTN"]:
    TARGETS[g] = consequence_targets(df, g, ratio=1.0)
    print(f"{g:7s} {TARGETS[g]}")
print(f"\ntotal controls requested: {sum(sum(d.values()) for d in TARGETS.values())}")

from data_prep import build_full_dataset

t0 = time.time()

smoke = build_full_dataset(
    clinvar_dataset_path = clinvar_dataset_path,
    clinvar_raw_path     = clinvar_raw_path,
    gnomAD_csv_paths     = gnomAD_csv_paths,
    out_path             = smoke_path,
    faf_threshold        = 0.0,
    ratio                = 1.0,
    limit                = 20,
)

elapsed = time.time() - t0
n_new = int((smoke["source"] == "gnomAD").sum())
print(f"\nsmoke test took {elapsed/60:.1f} min, {n_new} of 60 rows survived")



t0 = time.time()

full = build_full_dataset(
    clinvar_dataset_path = clinvar_dataset_path,
    clinvar_raw_path     = clinvar_raw_path,
    gnomAD_csv_paths     = gnomAD_csv_paths,
    out_path             = out_path,
    faf_threshold        = 0.0,
    ratio                = 1.0,
)

print(f"\nfinished in {(time.time() - t0) / 60:.1f} minutes")


check = pd.read_csv(out_path)

print("rows:", len(check))
print(check["source"].value_counts().to_string(), "\n")

print("ref_sequence on every row :", bool(check["ref_sequence"].notna().all()))
print("mutant windows all 201    :", bool((check["sequence"].str.len() == 201).all()))
print("ref windows all 201       :", bool((check["ref_sequence"].str.len() == 201).all()))
print("no duplicate variants     :",
      not check.duplicated(subset=["chrom", "pos", "ref", "alt"]).any(), "\n")


print(check.groupby("source")["consequence"].value_counts().to_string(), "\n")


got = check[check["source"] == "gnomAD"].groupby(["gene", "consequence"]).size()

print(f"{'cell':22s} {'asked':>6s} {'got':>5s} {'lost':>5s}")
total_lost = 0
for gene, d in TARGETS.items():
    for cons, want in d.items():
        n = int(got.get((gene, cons), 0))
        total_lost += want - n
        print(f"{gene + '/' + cons:22s} {want:6d} {n:5d} {want - n:5d}")
        

print(pd.crosstab([check["gene"], check["consequence"]], check["label"]).to_string())



In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Finds the project folder and puts src first, so the current files are loaded
here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "src/data_prep.py").exists())
if str(project_root / "src") in sys.path:
    sys.path.remove(str(project_root / "src"))
sys.path.insert(0, str(project_root / "src"))
sys.modules.pop("model_extra_features", None)
import model_extra_features as features


full = pd.read_csv(project_root / "data/processed/dataset_full.csv")
matched = pd.read_csv(project_root / "data/processed/matched_seed42.csv")
ttn_controls = (full.gene.eq("TTN") & full.consequence.eq("nonsense") & full.label.eq("Benign")).sum()
print("dataset_full rows:", len(full), "| TTN nonsense controls:", ttn_controls)
print("matched rows:", len(matched), "| fingerprint ok:", int(matched["pos"].sum()) == 268677163703)

ann = features.annotate(matched)
span = {g: (d["pos"].min(), d["pos"].max()) for g, d in matched.groupby("gene")}
X1 = features.build(matched, ann = ann)                    
X2 = features.build(matched, ann = ann, gene_span = span)   
print("shapes:", X1.shape, X2.shape)
print("column 32:", X2.columns[32],)
print(X2.loc[[0, 717, 1406, 1403], ["rel_pos", "tx_protein_frac", "tx_intron_dist", "ttn_psi_dcm",
                                     "ttn_in_novex3", "aa_grantham", "aa_blosum"]].round(3))

In [ ]:
import sys
from pathlib import Path
import pandas as pd

here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "src/data_prep.py").exists())
if str(project_root / "src") in sys.path:
    sys.path.remove(str(project_root / "src"))
sys.path.insert(0, str(project_root / "src"))
for mod in ["train", "model", "model_extra_features"]:
    sys.modules.pop(mod, None)
import train

matched = pd.read_csv(project_root / "data/processed/matched_seed42.csv")

# Every row must get a fold, and every gene must be in every fold
tr, va, te, de = train.make_regions_split(matched, seed = 49, verbose = False)
data = pd.concat([tr, va, te], ignore_index = True)
fold, near_edge = train.block_folds(data, n_folds = 5, seed = 0)
print("rows with no fold:", int((fold == -1).sum()))
print(pd.crosstab(data["gene"], fold))


within, within_skip, binary = train.cross_validate(matched, split_seed = 49, cv_seeds = (0, 1, 2))

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "src/data_prep.py").exists())
if str(project_root / "src") in sys.path:
    sys.path.remove(str(project_root / "src"))
sys.path.insert(0, str(project_root / "src"))
for mod in ["train", "model", "model_extra_features", "encoding"]:
    sys.modules.pop(mod, None)
import train

matched = pd.read_csv(project_root / "data/processed/matched_seed42.csv")
import torch
from model import CardiacCNN
saved = torch.load(project_root / "results" / "final_models.pt")
models = []
for state in saved["states"]:
    m = CardiacCNN(n_features = state["features.0.weight"].shape[1])
    m.load_state_dict(state)
    m.eval()
    models.append(m)

# The same split and scaling the models were trained with
tr, va, te, de = train.make_regions_split(matched, seed = 49, verbose = False)
F_fit, F_test = train.make_features_tensors(pd.concat([tr, va], ignore_index = True), [te])
X_test, y_test = train.prepare_tensors(te)
probs = train.predict(models, X_test, F_test, "cpu")
yt, preds = y_test.numpy(), probs.argmax(axis = 1)

# ---- Locked test split (343 variants, one pass) ----
print(f"Test accuracy: {(preds == yt).mean() * 100:.1f}%")
print(f"Macro one-vs-rest AUC: {roc_auc_score(yt, probs, multi_class = 'ovr', average = 'macro'):.3f}")
print(f"Binary AUC: {train.binary_auc(yt, probs):.3f}")
mean, low, high = train.bootstrap_auc((yt != train.LABELS_TO_INDEX['Benign']).astype(int), 1 - probs[:, train.LABELS_TO_INDEX['Benign']])
print(f"Binary AUC 95% confidence interval: {low:.3f} to {high:.3f}")
print(classification_report(yt, preds, target_names = train.LABELS, zero_division = 0))
print(confusion_matrix(yt, preds))
train.within_stratum_auc(te, probs)

# ---- Cross-validation (CV seed 0: every non-demo variant predicted once, by a model that never saw it) ----
cv = pd.read_csv(project_root / "results" / "cv_predictions.csv")
cv0 = cv[cv["cv_seed"] == 0]
y_cv = cv0["label"].map(train.LABELS_TO_INDEX).to_numpy()
pred_cv = cv0[["p_hcm", "p_dcm", "p_benign"]].to_numpy().argmax(axis = 1)
print("\nCross-validation, CV seed 0, 1,721 held-out variants:")
print(classification_report(y_cv, pred_cv, target_names = train.LABELS, zero_division = 0))
print(confusion_matrix(y_cv, pred_cv))

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Finds the project root 
here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "src/data_prep.py").exists())
if str(project_root / "src") in sys.path:
    sys.path.remove(str(project_root / "src"))
sys.path.insert(0, str(project_root / "src"))

sys.modules.pop("Revel_cadd_bench", None)
import Revel_cadd_alphamissense_bench as bench

results = project_root / "results"
matched = pd.read_csv(project_root / "data/processed/matched_seed42.csv")
preds = pd.read_csv(results / "cv_predictions.csv")

# Tool scores: downloads once 
tools = bench.check_tool_scores(matched, bench.fetch_tool_scores(matched))

# Known-answer tests, then the comparison table 
bench.harness_self_test(preds)
table = bench.benchmark(preds, tools)
table.to_csv(results / "benchmark.csv", index = False)
print(table.drop(columns = ["variants", "model - tool"]).to_string(index = False))

In [ ]:
import torch

# Trains the 3 final models (train + val, seed 49 split), the same way train_model always does
models, (X_test, F_test, y_test), demo_df, test_df = train.train_model(matched, seed = 49, evaluate_test = False)

# Their predictions on the test split, saved too, so a future reload can be checked against them
probs = train.predict(models, X_test, F_test, "cpu")

out = project_root / "results" / "final_models.pt"
out.parent.mkdir(exist_ok = True)
assert not out.exists(), "final_models.pt already exists; rename it before saving new models"
torch.save({"states": [m.state_dict() for m in models], "test_probs": torch.tensor(probs)}, out)
print("saved 3 models to", out, "| test binary AUC", round(train.binary_auc(y_test.numpy(), probs), 3))

In [ ]:
from diagnose import run

run(out_path)

In [ ]:
import sys
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

out_path = str(data_path / "processed" / "dataset_full.csv")

from train import match_cells, make_regions_split, min_gap

matched = match_cells(pd.read_csv(out_path), ratio=1.0, seed=49, verbose=False)
train_df, val_df, test_df, demo_df = make_regions_split(matched, seed=49)
print("min gap test:", min_gap(test_df, train_df))

In [ ]:
import sys, json, importlib
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"
ann_path  = data_path / "annotations"

# Puts src first in the search list, always
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))

sys.modules.pop("get_annotations", None)
import get_annotations
print("using", get_annotations.__file__)

# Quick test on one page before the full download
import requests
test = get_annotations.parse_exon_page(
    requests.get("https://www.cardiodb.org/titin/titin_exon.php?id=327", timeout=20).text)
print("exon 327 test:", test)
assert test is not None, "regex matched nothing: change the coords pattern to start at 'Meta-Transcript Start End'"

# Full download (about 1 to 2 minutes)
ann_path.mkdir(parents=True, exist_ok=True)
get_annotations.download_exon_maps(str(ann_path))
get_annotations.download_ttn_exons(str(ann_path))

# Checks
for chrom in ["chr14", "chr11", "chr2"]:
    assert (ann_path / f"tx_{chrom}.json").exists(), f"tx_{chrom}.json missing"
ttn = json.load(open(ann_path / "ttn_exons.json"))
print("TTN exons saved:", len(ttn))
assert len(ttn) == 364, "expected 364 exons"
assert ttn["327"]["s38"] == 178576416 and ttn["327"]["psi_dcm"] == 100
print("saved to:", ann_path)

In [1]:
# Score and explain every ClinVar VUS of the 4 trained kinds, save vus_scores.csv, write 15 showcase reports
import sys, time
from pathlib import Path
import pandas as pd

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/data_prep.py").exists())
src_path, data_path, results_path = project_root / "src", project_root / "data", project_root / "results"
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))
for module in ["vus", "report", "explain"]:
    sys.modules.pop(module, None)
import explain, report, vus
start = time.time()

matched = pd.read_csv(data_path / "processed" / "matched_seed42.csv")
assert int(matched["pos"].sum()) == 268677163703, "matched_seed42.csv is not the file the models were trained on"
setup = explain.final_setup(matched)
models = explain.load_final_models(matched, setup)
full = pd.read_csv(data_path / "processed" / "dataset_full.csv")
cv = pd.read_csv(results_path / "cv_predictions.csv")

vus_df = vus.load_vus(full)
table, ig_vus, vus_setup, vus_ctx = vus.score_vus(vus_df, matched, models, cv, str(results_path))

print(table.groupby(["gene", "consequence"])["score"].describe().round(2).to_string())
flagged = table["warning_kinds"].fillna("").str.contains("limit|caution")
print("share with a limit/caution warning:", flagged.groupby([table["gene"], table["consequence"]]).mean().round(2).to_dict())

picks = report.pick_showcase(table)
rows = report.write_reports(vus_setup["extra_df"].loc[picks], ig_vus.loc[picks], vus_setup, vus_ctx,
                            str(results_path / "reports_vus"), show_truth = False, prefix = "V")
print(len(rows), "showcase VUS reports", f"({time.time() - start:.0f} s)")

c:\Users\jaira\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded models give the saved test predictions (largest difference 1.8e-07); test binary AUC 0.736, macro 0.860
Consequence check: my call matches ClinVar for 100.0% of 4061 coding VUS
4176 VUS ready: {'MYH7/missense': 2404, 'MYBPC3/missense': 1556, 'MYBPC3/noncoding': 123, 'TTN/nonsense': 93}
Read saved explanations from c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\results\ig_vus.csv
Saved 4176 rows to c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\src\..\data\external\tool_scores_vus.csv
Coverage (scored / total):
                        revel alphamissense       cadd
gene   consequence                                    
MYBPC3 missense     1556/1556     1556/1556  1556/1556
       noncoding        0/123         0/123      5/123
MYH7   missense     2404/2404     2402/2404  2404/2404
TTN    nonsense          0/93          0/93      93/93
Read saved tool scores from c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\src\..\data\processed

In [2]:
import sys, time
from pathlib import Path
import pandas as pd

def find_project_root(marker = "src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path     = project_root / "src"
results_path = project_root / "results"

# Puts src first in the search list, always, and reloads explain and report so edits are picked up
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))
for module in ["report", "explain"]:
    sys.modules.pop(module, None)
import explain, report
import Revel_cadd_alphamissense_bench as bench
print("using", report.__file__)
start = time.time()

# Same data and the same 3 saved models
matched = pd.read_csv(project_root / "data" / "processed" / "matched_seed42.csv")
assert int(matched["pos"].sum()) == 268677163703, "matched_seed42.csv is not the file the models were trained on"
setup = explain.final_setup(matched)
models = explain.load_final_models(matched, setup)

# Names must match ClinVar's names before they go on any report
my_names = report.hgvs_names(matched)
official = matched["name"].fillna("").str.extract(r":(c\.[^ ]+)")[0]
has = official.notna()
match = (official[has] == my_names.loc[has, "c"]).mean()
print(f"HGVS check: my c. names match ClinVar for {match:.1%} of {int(has.sum())} ClinVar variants")
assert match > 0.99, "computed names disagree with ClinVar; fix before printing names on reports"

# Context shared by every report, which is typical controls, reliability, calibration, tool scores, protein maps
preds = pd.read_csv(results_path / "cv_predictions.csv")
tools = bench.check_tool_scores(matched, bench.fetch_tool_scores(matched))
ctx = report.build_context(matched, setup, preds, tools)
print(f"near AUC {ctx['near_auc']:.3f}, far AUC {ctx['far_auc']:.3f}, TTN high-use controls {ctx['ttn_high_controls']} of {ctx['ttn_controls']}")

# IG for the 94 locked demo variants 
feature_names = list(setup["raw"](setup["test_df"]).columns)
typical = explain.typical_benign(setup["F_fit"], setup["fit_df"])
ig_demo = explain.explain_saved(models, setup["demo_df"], setup["F_demo"], typical, feature_names,
                                str(results_path / "ig_demo.csv"))

# One report per demo variant, plus index.html
out_dir = results_path / "reports_demo"
rows = report.write_reports(setup["demo_df"], ig_demo, setup, ctx, str(out_dir), prefix = "D")
print(f"{len(rows)} reports written to {out_dir}")
print(f"Open {out_dir / 'index.html'} in your browser to browse them. Done in {time.time() - start:.0f} s")

using c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\src\report.py
Loaded models give the saved test predictions (largest difference 1.8e-07); test binary AUC 0.736, macro 0.860
HGVS check: my c. names match ClinVar for 100.0% of 1178 ClinVar variants
Read saved tool scores from c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project\src\..\data\processed\tool_scores.csv
Coverage (scored / total):
                      revel alphamissense       cadd
gene   consequence                                  
MYBPC3 missense       46/46         46/46      46/46
       noncoding      0/204         0/204     85/204
       nonsense        0/20          0/20      20/20
MYH7   missense     422/422       422/422    422/422
       noncoding       0/18          0/18       7/18
       nonsense         0/6           0/6        6/6
TTN    nonsense      1/1374        0/1374  1373/1374
revel: blanked 1 score(s) on variants that are not missense in the MANE transcript:
     gene con

In [ ]:
from model import CardiacCNN
import torch

# Build a fresh, untrained model with the same architecture
loaded_model = CardiacCNN(seq_len=201, n_classes=3)

# Load the saved weights from Drive into it
loaded_model.load_state_dict(torch.load("/content/drive/MyDrive/model_state_backup_0.954.pt"))
loaded_model.eval()

print(loaded_model)